In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))
print(os.getcwd())  # confirm your location


C:\Users\Shooq\Employees-talent-analytics\notebooks


In [2]:
from src.data_loader import HRDataLoader
from src.validators import HRDataValidator

In [3]:
loader = HRDataLoader(config_path='../config.yaml')
datasets = loader.load_all()

INFO:src.data_loader:Config file loaded successfully
INFO:src.data_loader:Raw data path set to: C:\Users\Shooq\Employees-talent-analytics\data\raw
INFO:src.data_loader:Starting to load all datasets...
INFO:src.data_loader:Loaded employees.csv: 30 rows, 6 columns
INFO:src.data_loader:Loaded project_assignments.csv: 109 rows, 7 columns
INFO:src.data_loader:Loaded salary_history.csv: 45 rows, 6 columns
INFO:src.data_loader:Loaded performance_reviews.csv: 90 rows, 7 columns
INFO:src.data_loader:Loaded training_history.csv: 80 rows, 7 columns
INFO:src.data_loader:All datasets loaded successfully


In [4]:
for name, df in datasets.items():
    if df is not None:
        print(f"{name}: {df.shape}")
    else:
        print(f"{name}: Failed to load")

employees: (30, 6)
projects: (109, 7)
salary: (45, 6)
performance: (90, 7)
training: (80, 7)


In [5]:
validator = HRDataValidator()
validator.generate_report(datasets)


   HR DATA VALIDATION REPORT

 EMPLOYEES
   Status:  Valid

 PERFORMANCE
   Status:  Valid



In [6]:
print(datasets['training'].info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   training_id           80 non-null     int64 
 1   employee_id           80 non-null     int64 
 2   course_name           80 non-null     object
 3   course_category       80 non-null     object
 4   completion_date       80 non-null     object
 5   hours_invested        80 non-null     int64 
 6   certification_earned  80 non-null     bool  
dtypes: bool(1), int64(3), object(3)
memory usage: 4.0+ KB
None


In [7]:
print(datasets['performance'].columns.tolist())

['review_id', 'employee_id', 'review_quarter', 'technical_score', 'leadership_score', 'innovation_score', 'reviewer_notes']


In [8]:
print(datasets['projects'].head(5))

   assignment_id  employee_id                       project_name  \
0              1          101            Solar Integration Study   
1              2          101    Energy Demand Forecasting Model   
2              3          101    Energy Demand Forecasting Model   
3              4          101  Carbon Capture Analytics Platform   
4              5          101  Carbon Capture Analytics Platform   

   hours_worked  completion_rate  manager_rating assignment_date  
0           113             97.4            4.86      2024-01-30  
1            64             96.2            4.84      2023-10-25  
2            92             95.5            4.83      2024-01-28  
3            47             99.3            4.72      2024-01-12  
4            71             98.6            4.94      2023-11-19  


# Performance Calculator

In [9]:
from src.performance_calculator import PerformanceCalculator
calculator = PerformanceCalculator(loader.config)
df_performance = datasets['performance'].copy()
df_performance = calculator.compute_composite_score(df_performance)
print(df_performance[['employee_id', 'review_quarter', 'composite_score']].head())

INFO:src.performance_calculator:PerformanceCalculator initialized — high performer threshold: 4.5
INFO:src.performance_calculator:Composite scores computed for 90 records


   employee_id review_quarter  composite_score
0          101        2023-Q4         4.770000
1          101        2024-Q1         4.966667
2          101        2024-Q2         4.986667
3          102        2023-Q4         4.583333
4          102        2024-Q1         4.606667


In [10]:
high_performers = calculator.identify_high_performers(df_performance)
print(f"High Performers: {len(high_performers)} employees")
print(high_performers[['employee_id', 'average_score']])

INFO:src.performance_calculator:Identified 6 high performers


High Performers: 6 employees
    employee_id  average_score
0           101       4.907778
3           102       4.615556
6           103       4.786667
9           104       4.756667
12          105       4.742222
15          106       4.865556


In [11]:
declining = calculator.detect_declining_talent(df_performance)
print(f"\nDeclining Talent: {len(declining)} employees")
print(declining[['employee_id']])

INFO:src.performance_calculator:Identified 3 talents_drop



Declining Talent: 3 employees
    employee_id
84          129
85          129
86          129


# Talents scores

In [12]:
from src.talent_score import TalentScorer
talent = TalentScorer(loader.config)
df_training = datasets['training'].copy()

In [13]:
dev_score = talent.compute_development_score(df_training)

In [14]:
print(dev_score)

    employee_id  development_score
0           101                  6
6           102                  6
12          103                  5
17          104                  5
22          105                  5
27          106                  6
33          107                  3
36          108                  2
38          109                  3
41          110                  4
45          111                  2
47          112                  3
50          113                  3
53          114                  2
55          115                  3
58          116                  3
61          117                  2
63          118                  2
65          119                  4
69          120                  2
71          121                  3
74          122                  4
78          123                  2


In [15]:
growth_score = talent.compute_growth_score(df_performance)

INFO:src.talent_score:'growth scores computed for 90 records


In [16]:
print(growth_score.head())

   review_id  employee_id review_quarter  technical_score  leadership_score  \
0          1          101        2023-Q4             4.80              4.87   
1          2          101        2024-Q1             4.97              4.89   
2          3          101        2024-Q2             5.01              4.98   
3          4          102        2023-Q4             4.47              4.73   
4          5          102        2024-Q1             4.72              4.62   

   innovation_score                          reviewer_notes  composite_score  \
0              4.64  Exceeds expectations on complex tasks.         4.770000   
1              5.04  Exceeds expectations on complex tasks.         4.966667   
2              4.97  Exceeds expectations on complex tasks.         4.986667   
3              4.55  Exceeds expectations on complex tasks.         4.583333   
4              4.48  Exceeds expectations on complex tasks.         4.606667   

   average_score  growth_score  
0       4.9

In [17]:
promotion_score =talent.compute_promotion_score(df_performance, df_training)

INFO:src.talent_score:'growth scores computed for 90 records


In [18]:
print(promotion_score.head())

   employee_id  growth_score  development_score  performance_score  perf_norm  \
0          101      0.216667                6.0           4.907778   1.000000   
5          106      0.146667                6.0           4.865556   0.979359   
4          105      0.300000                5.0           4.742222   0.919066   
2          103      0.173333                5.0           4.786667   0.940793   
1          102      0.073333                6.0           4.615556   0.857143   

   growth_norm  dev_norm  promotion_score  
0     0.836601      1.00         0.950980  
5     0.699346      1.00         0.901548  
4     1.000000      0.75         0.892626  
2     0.751634      0.75         0.826807  
1     0.555556      1.00         0.809524  


In [19]:
print(datasets['employees'].columns.tolist())

['employee_id', 'full_name', 'office_location', 'department', 'hire_date', 'salary_sar']


In [20]:
print(datasets['projects'].columns.tolist())


['assignment_id', 'employee_id', 'project_name', 'hours_worked', 'completion_rate', 'manager_rating', 'assignment_date']


# Succession Analyzer

In [21]:
from src.succession_analyzer import SuccessionAnalyzer

In [22]:
succession = SuccessionAnalyzer(loader.config)

In [23]:
df_employees =datasets['employees'].copy()
df_ready = succession.identify_ready_now(df_employees, high_performers, df_training)

In [24]:
print(df_ready)

    employee_id
0           101
7           102
12          103
17          104
23          105
27          106


In [25]:
df_scc = succession.map_succession_pipeline(df_ready, df_employees)

In [26]:
print(df_scc)

    department office_location  succession_candidates_count   status
0    Analytics          Riyadh                          1.0  covered
1  Engineering          Riyadh                          2.0  covered
2  Engineering         Dhahran                          2.0  covered
3    Analytics          Jeddah                          1.0  covered
4   Operations         Dhahran                          0.0  at risk
5    Analytics         Dhahran                          0.0  at risk
6   Operations          Jeddah                          0.0  at risk
7   Operations          Riyadh                          0.0  at risk
8  Engineering          Jeddah                          0.0  at risk


In [27]:
print(f"Total employees: {len(datasets['employees'])}")
print(f"Total high performers: {len(high_performers)}")
print(f"Total ready-now candidates: {len(df_ready)}")
print(datasets['employees']['department'].value_counts())
print(datasets['employees']['office_location'].value_counts())

Total employees: 30
Total high performers: 6
Total ready-now candidates: 6
department
Analytics      15
Engineering     9
Operations      6
Name: count, dtype: int64
office_location
Dhahran    12
Riyadh     11
Jeddah      7
Name: count, dtype: int64


# API Client

In [28]:
from src.APIclient import HRApiClient
from src.report_generator import HRReportGenerator
from src.succession_analyzer import SuccessionAnalyzer

client = HRApiClient(base_url='https://api.exchangerate-api.com/v4')

C:\Users\Shooq\anaconda3\envs\Employees-Analytics\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
INFO:src.APIclient:HRApiClient initialized for base URL: https://api.exchangerate-api.com/v4 (Timeout: 10s)


In [29]:
# Test _make_request directly
client = HRApiClient(base_url='https://api.exchangerate-api.com/v4')


INFO:src.APIclient:HRApiClient initialized for base URL: https://api.exchangerate-api.com/v4 (Timeout: 10s)


In [32]:
rate = client.get_exchange_rate()
print(f"Rate: {rate}")

INFO:src.APIclient:Making GET request to https://api.exchangerate-api.com/v4/latest/USD (Attempt 1/2)


Rate: 3.75


In [30]:
# Cell 11 — Live data
rate = client.get_exchange_rate()
print(f"Current USD/SAR Rate: {rate}")

market = client.get_labor_market_index()
print(f"Saudi Labor Market: {market}")

INFO:src.APIclient:Making GET request to https://api.exchangerate-api.com/v4/latest/USD (Attempt 1/2)
INFO:src.APIclient:Making GET request to https://api.worldbank.org/v2/country/SAU/indicator/SL.UEM.TOTL.ZS (Attempt 1/2)


Current USD/SAR Rate: 3.75
Saudi Labor Market: {'country': 'Saudi Arabia', 'year': '2025', 'saudi_unemployment_rate': 3.038, 'source': 'World Bank '}


In [33]:
# Cell 12
config = loader.config
reporter = HRReportGenerator(config)

summary = reporter.generate_summary_stats(datasets, df_performance)

talent_report = reporter.generate_talent_report(
    high_performers=high_performers,
    declining=declining,
    promotion_ranking=promotion_score
)

reporter.print_executive_report({**summary, **talent_report})

INFO:src.report_generator:Executive report printed successfully


HR TALENT ANALYTICS EXECUTIVE REPORT
Generated: 2026-06-02 15:48:18

 WORKFORCE OVERVIEW
Total Employees        : 30
Project Assignments    : 109
Training Records       : 80
Avg Performance Score  : 3.89

 TALENT HIGHLIGHTS
  High Performers        : 6 employees
  Declining Talent       : 1 employees

 TOP 5 PROMOTION CANDIDATES
  Rank 1  |  Employee 101  |  Score: 0.951
  Rank 2  |  Employee 106  |  Score: 0.902
  Rank 3  |  Employee 105  |  Score: 0.893
  Rank 4  |  Employee 103  |  Score: 0.827
  Rank 5  |  Employee 102  |  Score: 0.810



In [35]:
# Cell 13 — Succession Pipeline
analyzer = SuccessionAnalyzer(config)

df_ready = analyzer.identify_ready_now(
    df_employees, high_performers, df_training
)
print(f"Ready-Now Succession Candidates: {len(df_ready)}")
print(df_ready)

pipeline = analyzer.map_succession_pipeline(df_ready, datasets['employees'])
print("\nSuccession Coverage by Department & Office:")
print(pipeline)

Ready-Now Succession Candidates: 6
    employee_id
0           101
7           102
12          103
17          104
23          105
27          106

Succession Coverage by Department & Office:
    department office_location  succession_candidates_count   status
0    Analytics          Riyadh                          1.0  covered
1  Engineering          Riyadh                          2.0  covered
2  Engineering         Dhahran                          2.0  covered
3    Analytics          Jeddah                          1.0  covered
4   Operations         Dhahran                          0.0  at risk
5    Analytics         Dhahran                          0.0  at risk
6   Operations          Jeddah                          0.0  at risk
7   Operations          Riyadh                          0.0  at risk
8  Engineering          Jeddah                          0.0  at risk
